# GR-Conditioned Output-Transformer LSTM — Grey Box + continuous TFiLM (Diff-SSL)

**Google Colab**: Runtime → **GPU**. Open via *File → Open notebook → GitHub*
(`5aola/Virtual-Analogue-Compressor-Modelling`); cell 1 clones the repo for the
`06_output` modules and mounts Drive for the dataset. **Push local changes before running.**

## Idea — grey-box envelope + a learned colorist *that knows how hard the comp is working*

Same grey-box decomposition as `train_lstm_output_transformer.ipynb`, **plus** the
exported GR curve is fed back in as a **continuous, time-varying conditioning
signal** (not the 4 static knobs):

```
dry ─(× exported GR gain)─► amplitude-matched input ─►┐
                                                       ├─► [GR-conditioned LSTM] ─► wet
exported GR curve (dB) ───────────────────────────────┘   (TFiLM + concat)
```

1. **Amplitude matching** (unchanged): `amp = dry · 10**(gr_db/20)` reproduces the
   compressor's level/dynamics from the known GR envelope (≈2 % RMS env. error).
2. **GR conditioning** (new): the same `gr_db` curve also conditions the LSTM, so
   the colorist is told the **instantaneous compression depth** — which governs how
   much nonlinear coloration (harmonics, transient shaping) to add.

## Conditioning strategy — why *time-varying* FiLM, and where it comes from

`05_conditioning` conditioned on the **static** 4 knobs via `nablafx` **`TFiLM`**
(`cond=[B,4]`, broadcast over blocks). Here the conditioning is the GR curve, a
**dynamic** signal — so we use the nablafx **time-varying** path.

nablafx/Optical-DRC's `cond_type="tvcond"` (`nablafx/processors/lstm.py`) uses
`TVFiLMCond` to *learn* a dynamic conditioning sequence (a block-rate LSTM over
`|x|`) and `TVFiLMMod` to apply it. **`TVFiLMCond` only exists because the SOTA has
no ground-truth envelope.** We already have it — the exported GR curve **is** that
sequence — so we drop the learned generator and feed the real GR straight into
**`TVFiLMMod`**. Two complementary injection points (both flag-gated, default on):

| | mechanism | what it does |
|---|---|---|
| **`use_tvfilm`** | `TVFiLMMod` (nablafx) on the LSTM hidden features | block-wise affine modulation from the GR curve (+its block delta ≈ attack/release rate) — the headline **TFiLM** |
| **`concat_gr`** | sample-rate GR appended to the LSTM input | nablafx `tvcond` injection point — makes the recurrence itself GR-aware |

`TVFiLMMod` is **stateless** (1×1-conv adaptor + block affine), so only the two
main LSTM states are carried across TBPTT chunks — no extra conditioning state.

**Baseline-start preserved**: `dense_out` stays zero-init, so the residual
correction is exactly 0 at step 0 *regardless of FiLM scaling* — the net starts at
the validated amplitude-matched signal and only learns GR-gated coloration.

## What is identical to `train_lstm_output_transformer.ipynb`
- **Dataset / split**: Diff-SSL-G-Comp, all 10 settings × 10 songs, `splits.py`
  seed 42 (val = 1 song × all settings, test = held-out songs × lowest-threshold).
- **Loss / metrics**: `0.5·L1 + 0.5·MR-STFT` (+opt. ESR); ESR/RMSE/MAE/MSE.
- **Stateful TBPTT** over `segment_len` chunks, states carried chunk→chunk.


In [ ]:
# -- 0. Dependencies ---------------------------------------------------
# This variant uses nablafx (TVFiLMMod). Pin numpy first so lightning/nablafx
# installs can't downgrade Colab's numpy 2.x and break torch. Install
# lightning/nablafx --no-deps so they can't clobber Colab's CUDA torch.
# `rational` / `frechet_audio_distance` are nablafx import-chain deps we never
# use here; stub both so `from nablafx...` doesn't drag in broken wheels.
!pip install -q "numpy>=2.0,<2.6"
!pip install -q torchmetrics soundfile auraloss einops lightning-utilities packaging
!pip install -q --no-deps lightning nablafx

import sys, types

rational = types.ModuleType("rational")
rational.torch = types.ModuleType("rational.torch")
rational.torch.Rational = type("Rational", (), {})
sys.modules["rational"], sys.modules["rational.torch"] = rational, rational.torch

fad = types.ModuleType("frechet_audio_distance")
fad.FrechetAudioDistance = type("FrechetAudioDistance", (), {})
sys.modules["frechet_audio_distance"] = fad

import numpy as np, torch
assert np.__version__.startswith("2."), f"numpy {np.__version__} - restart runtime, re-run cell 0"
print(f"numpy {np.__version__}, torch {torch.__version__}")


In [ ]:
# -- 1. Mount Drive (dataset) + clone repo from GitHub (code) ---------
# The repo is NOT synced to Drive (only data/ is). Code comes from GitHub -
# push local changes before (re)running this cell; re-running pulls updates.

import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ROOT = "/content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp"
REPO_URL = "https://github.com/5aola/Virtual-Analogue-Compressor-Modelling.git"
REPO_ROOT = "/content/Virtual-Analogue-Compressor-Modelling"

if os.path.isdir(REPO_ROOT):
    !git -C "{REPO_ROOT}" pull --ff-only
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_ROOT}"

DATA_ROOT = DRIVE_DATA_ROOT

# Module directory for this notebook.
# NOTE: Colab clones from GitHub, so any *uncommitted* local folders won't exist there.
# The implementation for this notebook lives in `06_output/`.
CANDIDATE_DIRS = [
    os.path.join(REPO_ROOT, "06_output"),
    os.path.join(REPO_ROOT, "06_conditioning"),  # legacy / local-only fallback
]
COND_DIR = next((d for d in CANDIDATE_DIRS if os.path.isfile(os.path.join(d, "dataset.py"))), None)
assert COND_DIR is not None, (
    f"Could not find dataset.py in any of: {CANDIDATE_DIRS}. "
    f"Did the clone succeed? REPO_ROOT={REPO_ROOT}"
)

OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), "output_transformer_tfilm_runs")

assert os.path.isdir(os.path.join(DATA_ROOT, "gr_curves")), f"Bad DATA_ROOT: {DATA_ROOT}"
assert os.path.isdir(os.path.join(DATA_ROOT, "processed_ground_truth")), "Missing wet audio dir"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# repo root (for `src`) + module dir (for dataset/model/system/splits)
for p in (REPO_ROOT, COND_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"COND_DIR   : {COND_DIR}")
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")


In [ ]:
# -- 2. Cache dataset to Colab local SSD ------------------------------
# The output transformer needs dry + GR curves (to build the matched input)
# AND the wet audio (the target). Mirror 05's cache and add the wet WAVs.

import shutil
from dataset import discover_output_transformer_pairs

LOCAL_DATA_ROOT = "/content/Diff-SSL-G-Comp"

pairs = discover_output_transformer_pairs(DATA_ROOT)
settings = sorted({p["setting"] for p in pairs})
songs = sorted({p["song"] for p in pairs})
print(f"Caching {len(songs)} songs x {len(settings)} settings ({len(pairs)} pairs) -> {LOCAL_DATA_ROOT}")

def _mirror(src: Path, dst: Path):
    src, dst = Path(src), Path(dst)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

# dry WAVs (one per song, shared across settings)
for song in songs:
    fn = f"{song}_UnmasteredWAV.wav"
    _mirror(Path(DATA_ROOT) / "processed_normalized" / fn,
            Path(LOCAL_DATA_ROOT) / "processed_normalized" / fn)

# GR curves (.pt) + wet WAVs (-exported.wav), per (song, setting) pair
for p in pairs:
    _mirror(p["gr"], Path(LOCAL_DATA_ROOT) / "gr_curves" / p["setting"] / Path(p["gr"]).name)
    _mirror(p["wet"], Path(LOCAL_DATA_ROOT) / "processed_ground_truth" / p["setting"] / Path(p["wet"]).name)

DATA_ROOT = LOCAL_DATA_ROOT
print(f"Using local cache: {DATA_ROOT}")


In [ ]:
# -- 3. Imports & hyper-parameters ------------------------------------

import json
from datetime import datetime

import torch
import lightning as pl
from lightning.pytorch.callbacks import (
    EarlyStopping, LearningRateMonitor, ModelCheckpoint, TQDMProgressBar,
)
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

from dataset import SAMPLE_RATE, SEGMENT_LEN, WINDOW, discover_output_transformer_pairs
from dataset_tfilm import OutputTransformerTFiLMDataModule
from model_tfilm import OutputTransformerTFiLMLSTM
from system_tfilm import OutputTransformerTFiLMSystem
from splits import build_split_manifest
from amplitude_match import GR_DB_MIN, GR_DB_MAX

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "WARNING: CPU runtime")

# -- split (identical to 05 / 06) --
SPLIT_SEED   = 42
N_VAL_SONGS  = 1
N_TEST_SONGS = 2

# -- training --
LR                  = 1e-3
MAX_EPOCHS          = 800       # committed budget == CosineAnnealingLR T_max
EARLY_STOP_PATIENCE = 200
SCHEDULER           = "cosine"  # or "plateau" (ReduceLROnPlateau, nablafx recipe)
ETA_MIN             = 1e-6
WARMUP_SAMPLES      = 0         # drop N samples on each track's fresh-state chunk

# -- model (GR-conditioned sample-rate windowed LSTM) --
ENCODER_CHANNELS = 8
HIDDEN_SIZE      = 16
MID_CHANNELS     = 8
NUM_LSTM_LAYERS  = 2
OUTPUT_MODE      = "residual_add"   # "residual_gain" | "direct" for ablation
OUT_ACTIVATION   = "none"           # "tanh" to bound the output/correction

# -- GR conditioning (the new part) --
CONCAT_GR         = True    # append sample-rate GR to the LSTM input (tvcond-style)
USE_TVFILM        = True    # block-wise TVFiLMMod modulation from the GR curve
TVFILM_BLOCK_SIZE = 256     # samples/block -> 256/44100 ~= 5.8 ms modulation rate
GR_COND_CHANNELS  = 8       # GR embedding width feeding the TVFiLM adaptor
GR_USE_DELTA      = True    # also feed the block-to-block GR delta (attack/release)

# -- loss (SOTA waveform recipe) --
L1_WEIGHT     = 0.5
MRSTFT_WEIGHT = 0.5
ESR_WEIGHT    = 0.0   # set > 0 to add the Optical-DRC / comparative-study ESR term

RUN_TAG    = "lstm_output_transformer_gr_tfilm"
RESUME_RUN = None


In [ ]:
# -- 4. Preview split (must match 05) ---------------------------------

preview = build_split_manifest(
    discover_output_transformer_pairs(DATA_ROOT),
    seed=SPLIT_SEED, n_val_songs=N_VAL_SONGS, n_test_songs=N_TEST_SONGS,
)
print(f"Settings ({len(preview.all_settings)}): {preview.all_settings}")
print(f"Test settings (lowest T): {preview.test_settings}")
print(f"Train songs: {preview.train_songs}")
print(f"Val songs  : {preview.val_songs}")
print(f"Test songs : {preview.test_songs}")
print(f"Pairs - train={len(preview.train_pair_keys)} "
      f"val={len(preview.val_pair_keys)} test={len(preview.test_pair_keys)}")


In [ ]:
# -- 5. Model size ----------------------------------------------------

model = OutputTransformerTFiLMLSTM(
    window=WINDOW, encoder_channels=ENCODER_CHANNELS, hidden_size=HIDDEN_SIZE,
    mid_channels=MID_CHANNELS, num_lstm_layers=NUM_LSTM_LAYERS,
    output_mode=OUTPUT_MODE, out_activation=OUT_ACTIVATION,
    concat_gr=CONCAT_GR, use_tvfilm=USE_TVFILM,
    tvfilm_block_size=TVFILM_BLOCK_SIZE, gr_cond_channels=GR_COND_CHANNELS,
    gr_use_delta=GR_USE_DELTA,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"OutputTransformerTFiLMLSTM: {n_params:,} params  "
      f"(concat_gr={CONCAT_GR}, use_tvfilm={USE_TVFILM}, output_mode={OUTPUT_MODE})")
for name, mod in model.named_children():
    print(f"  {name:14s} {sum(p.numel() for p in mod.parameters()):,}")
print(f"\nWindow {WINDOW} samples | segment {SEGMENT_LEN} ({SEGMENT_LEN/SAMPLE_RATE:.2f}s) | "
      f"{SAMPLE_RATE} Hz | TVFiLM block {TVFILM_BLOCK_SIZE} "
      f"({TVFILM_BLOCK_SIZE/SAMPLE_RATE*1e3:.1f} ms)")


In [ ]:
# -- 6. Train ---------------------------------------------------------

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

assert DATA_ROOT.startswith("/content/"), "Run the cache cell first (cell 2)."

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    print(f"RESUMING: {RUN_NAME}")
else:
    RUN_NAME = f"ot_lstm_{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
split_path = os.path.join(RUN_DIR, "split_manifest.json")

dm = OutputTransformerTFiLMDataModule(
    data_root=DATA_ROOT, segment_len=SEGMENT_LEN, window=WINDOW,
    sample_rate=SAMPLE_RATE, split_seed=SPLIT_SEED,
    n_val_songs=N_VAL_SONGS, n_test_songs=N_TEST_SONGS,
    split_manifest_path=split_path,
)
dm.setup()
print(f"Train/val/test streams: {dm.train_dataset.B} / {dm.val_dataset.B} / {dm.test_dataset.B}")
print(f"Steps/epoch (train): {len(dm.train_dataset)}")

with open(os.path.join(RUN_DIR, "hparams.json"), "w") as f:
    json.dump({
        "approach": "gr_output_transformer_amplitude_matched_gr_conditioned",
        "model_type": "sample_rate_windowed_lstm",
        "source_model": "Optical-DRC create_model_LSTM (no conditioning) + PReLU front-end",
        "dataset": "Diff-SSL-G-Comp", "settings": "all 10 (pooled, no conditioning)",
        "input": "amplitude_matched = dry * 10**(clamp(gr_db, %g, %g)/20)" % (GR_DB_MIN, GR_DB_MAX),
        "conditioning": "exported GR curve as continuous TVFiLMMod + LSTM-input concat",
        "target": "wet audio (direct)", "sample_rate": SAMPLE_RATE,
        "window": WINDOW, "segment_len": SEGMENT_LEN,
        "split_seed": SPLIT_SEED, "train_songs": dm.split.train_songs,
        "val_songs": dm.split.val_songs, "test_songs": dm.split.test_songs,
        "test_settings": dm.split.test_settings,
        "model": {"encoder_channels": ENCODER_CHANNELS, "hidden_size": HIDDEN_SIZE,
                   "mid_channels": MID_CHANNELS, "num_lstm_layers": NUM_LSTM_LAYERS,
                   "output_mode": OUTPUT_MODE, "out_activation": OUT_ACTIVATION,
                   "concat_gr": CONCAT_GR, "use_tvfilm": USE_TVFILM,
                   "tvfilm_block_size": TVFILM_BLOCK_SIZE,
                   "gr_cond_channels": GR_COND_CHANNELS, "gr_use_delta": GR_USE_DELTA,
                   "num_params": n_params},
        "loss": {"l1": L1_WEIGHT, "mrstft": MRSTFT_WEIGHT, "esr": ESR_WEIGHT,
                  "ref": "nablafx-diffssl 0.5*L1+0.5*MR-STFT (+opt ESR)"},
        "metrics": ["esr", "rmse", "mae", "mse"],
        "lr": LR, "max_epochs": MAX_EPOCHS, "scheduler": SCHEDULER,
        "eta_min": ETA_MIN, "warmup_samples": WARMUP_SAMPLES,
        "early_stop_patience": EARLY_STOP_PATIENCE,
    }, f, indent=2)

system = OutputTransformerTFiLMSystem(
    model=model, lr=LR, l1_weight=L1_WEIGHT, mrstft_weight=MRSTFT_WEIGHT,
    esr_weight=ESR_WEIGHT, warmup_samples=WARMUP_SAMPLES, scheduler=SCHEDULER,
    max_epochs=MAX_EPOCHS, eta_min=ETA_MIN,
)


class ResumeOverrides(pl.Callback):
    # On resume, checkpoint restore overwrites cosine T_max and EarlyStopping
    # state with the old run's values - re-apply the notebook hparams so an
    # extended budget and a fresh early-stop window actually take effect.

    def on_train_start(self, trainer, pl_module):
        sched = trainer.lr_scheduler_configs[0].scheduler
        if hasattr(sched, "T_max"):
            sched.T_max = MAX_EPOCHS
        for cb in trainer.callbacks:
            if isinstance(cb, EarlyStopping):
                cb.patience = EARLY_STOP_PATIENCE
                cb.wait_count = 0
                cb.best_score = torch.tensor(float("inf"))


ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
callbacks = [
    ModelCheckpoint(dirpath=ckpt_dir, monitor="loss/val", mode="min", save_top_k=3,
                    save_last=True, filename="best-{epoch:03d}-{step}",
                    auto_insert_metric_name=False),
    LearningRateMonitor(logging_interval="epoch"),
    EarlyStopping(monitor="loss/val", mode="min", patience=EARLY_STOP_PATIENCE, verbose=True),
    TQDMProgressBar(refresh_rate=10),
    ResumeOverrides(),
]
loggers = [
    TensorBoardLogger(save_dir=RUN_DIR, name="tb", version=""),
    CSVLogger(save_dir=RUN_DIR, name="csv", version=""),
]

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS, accelerator="gpu", devices=1,
    precision="32-true",            # tiny model; fp32 keeps stateful carry + MR-STFT stable
    callbacks=callbacks, logger=loggers,
    gradient_clip_val=1.0, gradient_clip_algorithm="norm",
    log_every_n_steps=10, use_distributed_sampler=False,
)
trainer.fit(system, dm, ckpt_path=_resume_ckpt)
print(f"Best val loss: {callbacks[0].best_model_score:.6f}")
print(f"Best ckpt    : {callbacks[0].best_model_path}")


In [ ]:
# -- 7. Test (held-out songs x lowest-threshold settings) -------------

trainer.test(system, datamodule=dm, ckpt_path=callbacks[0].best_model_path)


In [ ]:
# -- 8. Plot: amplitude-matched baseline vs prediction vs target ------
# Streams the val set in order so the LSTM state settles, then plots a
# mid-track chunk per stream. The amplitude-matched input is the grey-box
# baseline; the gap from it to the target is what the GR-conditioned
# transformer learns.

import matplotlib.pyplot as plt
import numpy as np
from system import esr_metric, _detach_state

best = torch.load(callbacks[0].best_model_path, map_location="cuda", weights_only=False)
system.load_state_dict(best["state_dict"])
system.eval().cuda()
print(f"Loaded best checkpoint: {callbacks[0].best_model_path}")

val_steps = list(dm.val_dataloader())
pick = len(val_steps) // 2

state = None
with torch.no_grad():
    for s, (inp, gr, wet, mask, reset) in enumerate(val_steps):
        if bool(reset):
            state = None
        pred, state = system.model(inp.cuda(), gr.cuda(), state, return_state=True)
        state = _detach_state(state)
        if s == pick:
            amp = inp[:, :, WINDOW - 1:].cpu().numpy()   # matched-input baseline
            pred_np = pred.cpu().numpy()
            wet_np = wet.numpy()
            rows = torch.nonzero(mask).squeeze(1).tolist()
            break

n_plots = min(4, len(rows))
fig, axes = plt.subplots(n_plots, 1, figsize=(14, 3 * n_plots), sharex=True, squeeze=False)
t = np.arange(wet_np.shape[-1]) / SAMPLE_RATE
for ax, r in zip(axes[:, 0], rows[:n_plots]):
    ax.plot(t, amp[r, 0], label="Amplitude-matched (baseline in)", alpha=0.4, lw=0.5, color="gray")
    ax.plot(t, wet_np[r, 0], label="Target (wet)", alpha=0.8, lw=0.5)
    ax.plot(t, pred_np[r, 0], label="Predicted", alpha=0.8, lw=0.5)
    base_mae = float(np.mean(np.abs(amp[r, 0] - wet_np[r, 0])))
    pred_mae = float(np.mean(np.abs(pred_np[r, 0] - wet_np[r, 0])))
    pv = torch.from_numpy(pred_np[r]); tv = torch.from_numpy(wet_np[r])
    c = dm.val_dataset.cache[r]
    ax.set_title(f"{c['song']} / {c['setting']} (chunk {pick}) - "
                 f"MAE: baseline {base_mae:.4f} -> pred {pred_mae:.4f} | ESR {float(esr_metric(tv, pv)):.4f}")
    ax.set_ylabel("amp"); ax.legend(loc="lower right", fontsize=8); ax.set_ylim(-1.05, 1.05)
axes[-1, 0].set_xlabel("Time (s)")
fig.suptitle(f"GR-conditioned output-transformer LSTM - best val loss {callbacks[0].best_model_score:.6f}", y=1.005)
fig.tight_layout()
plot_path = os.path.join(RUN_DIR, "eval_output_comparison.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {plot_path}")
plt.show()


In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/tb"
